![](./radar_eval/gui/resources/pics/Logo_Sykno.svg "")

# __Radar Data Evaluation__ - ___Jupyter Notebook___

---
### __Introduction__

**Radar Data Measurement:**

- Radar Measurements Path (HDF5 Files): `/project_root/radar_eval/measurement/hdf5/`
- Radar HDF5 Controller for interaction via Radar Eval GUI or within the Jupyter Notebook: `/project_root/radar_eval/measurement/hdf5_controller.py` (see Radar HDF5 Controller Usage)
- The Radar radar data is a Numpy array `radar_data_cube` with a shape of `(n_frames, n_samples_per_chirp, n_rx, n_tx, n_shape_sets)` and its `dtype` is `float32`.


**Radar HDF5 Structure:**
- __/__
  - __Data__
    - ***Frame_Data_Cube_XXXX_ZZZZ***
      - `@delta_time`: `np.float32`
      - `@shape`: `(n_samples_per_chirp, n_rx, n_tx, n_shapes)`
      - `@timestamp`: `np.float32`
      - ...
  - __Metadata__
    - ***radar_bgt_reg_content***
      - `@radar_bgt_reg_content`:
    - ***radar_bgt_reg_content_readable***
      - `@radar_bgt_reg_content_readable`:
    - ***radar_config***
      - `@radar_config`:
    - ***radar_radar_parameters*** 
      - `radar_radar_parameters`

***Frame_Data_Cube_XXXX_ZZZZ*** where ***XXXX*** are multiples of `4095` and ***ZZZZ*** is the frame counter with values in the range $[0, 4095]$.

- ***XXXX*** $\in \{k \cdot 4095 \mid k \in \mathbb{N}, 0 \leq k \leq 9999\}$
- ***ZZZZ*** $\in \{0, 1, 2, \ldots, 4095\}$


**Radar data cube structure:**
| Dimension            | Size |
|----------------------|------|
| Number of samples    | n_samples_per_chirp $\in \{2^n \mid n \in \mathbb{N}, 6 \leq n \leq 11\}$ |
| Number of RX antennas| n_rx_active_antennas $\in \{n \mid n \in \mathbb{N}, 1 \leq n \leq 4\}$ |
| Number of TX antennas| n_tx_active_antennas $\in \{n \mid n \in \mathbb{N}, 1 \leq n \leq 2\}$ |
| Number of shape sets | n_shapes $\in \{n \mid n \in \mathbb{N}, 1 \leq n \leq 4095\}$ |
| Number of frames     | n_frames $\in \{n \mid n \in \mathbb{N}, 0 \leq n \leq 4095\}$ |


**Chirp data structure:**

| Samples      | ADC Channel 1 (RX1) |ADC Channel 2 (RX2) |ADC Channel 3 (RX3) |ADC Channel 4 (RX4) |
|-----------------|------------------------------|-----------------|------------------------------|-----------------|
| 0         | dtype=np.uint16 | dtype=np.uint16 | dtype=np.uint16 | dtype=np.uint16 |
| 1         | 12 bit ADC | 12 bit ADC | 12 bit ADC | 12 bit ADC |
| 2         | 0 - 1.2 V | 0 - 1.2 V | 0 - 1.2 V | 0 - 1.2 V |
| ...             | ...                          | ...                          | ...                          | ...                          |
| n-1         | 0 - 4095 | 0 - 4095 | 0 - 4095 | 0 - 4095 |


**Working with HDF5:**

- List measurement structure:
```sh 
    h5ls -r radar_hdf5_filename.hdf5
```
- Inspect a specific group or dataset:
```sh
    h5dump -d /Data/Frame_Data_Cube_XXXX_ZZZZ radar_hdf5_filename.hdf5
```
- Inspect measurement meta data:
```sh
    h5dump -g /Metadata radar_hdf5_filename.hdf5
```
- Get statistics e.g. number of groups, datasets, and attributes:
```sh
    h5stat radar_hdf5_filename.hdf5
```


In [ ]:
description = "Radar Root is: "
radar_root = !pwd
print(f"{description}{radar_root[0]}")
radar_meas_hdf5_path = f'{radar_root[0]}/radar_eval/measurement/hdf5/'
!tree -C {radar_meas_hdf5_path}

In [ ]:
radar_meas_hdf5_file_path = f'{radar_meas_hdf5_path}/MiRa6024I1A_Measurement_23_07_2024_18_21_15_Default_Session.hdf5'

In [ ]:
!h5ls -r {radar_meas_hdf5_file_path}

In [ ]:
!h5dump -d /Data/Frame_Data_Cube_0000_0000 {radar_meas_hdf5_file_path}

In [ ]:
!h5dump -g /Metadata {radar_meas_hdf5_file_path}

In [ ]:
!h5stat {radar_meas_hdf5_file_path}

___
### __Radar Data Evaluation__ - ___Python Code___


#### ***Radar HDF5 Controller Usage***


In [ ]:
import os
import numpy as np
import ipywidgets as widgets
from radar_eval.measurement.hdf5_controller import MIRA_HDF5_CTRL
from radar_eval.radar_system.radar_system_definition import MIRA_RADAR_PARAMETER
radar_meas_hdf5_path = './radar_eval/measurement/hdf5/'

# List all files in the folder
file_names = os.listdir(radar_meas_hdf5_path)

# Filter out non-file entries if needed
file_names = [f for f in file_names if os.path.isfile(os.path.join(radar_meas_hdf5_path, f)) and f != '.gitkeep']
sorted_file_names = sorted(file_names)

# Dropdown for file selection
file_dropdown = widgets.Dropdown(
    options=file_names,
    description=f'Select a File:',
)
print((f'Radar HDF5 Files in {radar_meas_hdf5_path}:'))
file_dropdown.layout.width = '650px'  # Adjust the width as needed
display(file_dropdown)



In [ ]:
from pathlib import Path
radar_hdf5_controller = MIRA_HDF5_CTRL(Path(radar_meas_hdf5_path+str(file_dropdown.value)).resolve())
data = radar_hdf5_controller.read_dataset("/Data/Frame_Data_Cube_0000_0000")
print(data)

In [ ]:
radar_hdf5_controller.print_hdf5_structure()

In [ ]:
stats = radar_hdf5_controller.get_dataset_statistics("/Data/Frame_Data_Cube_0000_0001")
print(stats)

In [ ]:
info = radar_hdf5_controller.get_dataset_info("/Data/Frame_Data_Cube_0000_0000")
print(info)

In [ ]:
frame_data_cube = radar_hdf5_controller.get_dataset_slice("/Data/Frame_Data_Cube_0000_0000")

In [ ]:
metadata = radar_hdf5_controller.read_metadata()
print(metadata)

In [ ]:
radar_param: MIRA_RADAR_PARAMETER = radar_hdf5_controller.unpickle_metadata()
print(radar_param)

In [ ]:
radar_data_cube = np.array(radar_hdf5_controller.load_radar_data_cube(), dtype=np.float32)
print(radar_data_cube.shape, radar_data_cube.dtype)

---
#### ***Inspect radar dataset***

1. **Load python packages and radar HDF5 measurement file**


2. **Open Radar HDF5 measurement file, load meta data and build radar data cube**

3. **Radar HDF5 measurement - data profiling**

    __Inspect Radar Radar System Parameters__
    - __radar_param__: meta-class contains __All__ other sub-classes
        - __sys__: sub-class contains __Radar__ specific values
        - __dsp__: sub-class contains __DSP__ parameters
        - __mon__: sub-class to __Monitor__ frame count, duration time, etc.
        - __gui__: sub-class contains __GUI__ relevant parameters
        - __meas__: sub-class contains __Measurement__ relevant parameters
        - __remt__: sub-class contains __Remote Control__ relevant parameters
        - __rply__: sub-class contains __Replay Control__ relevant parameters 

In [ ]:
print(f"{radar_param.__dict__=}")
print(f"{radar_param.dsp.__dict__=}")
print(f"{radar_param.gui.__dict__=}")
print(f"{radar_param.mon.__dict__=}")
print(f"{radar_param.sys.__dict__=}")
print(f"{radar_param.meas.__dict__=}")
print(f"{radar_param.remt.__dict__=}")
print(f"{radar_param.rply.__dict__=}")

3. **Radar HDF5 measurement - data profiling**

    1. __Check Radar System Parameters__
    - __radar_param__: meta-class contains __All__ other sub-classes

In [ ]:
print(f"{radar_data_cube.shape=}")
print(f"{frame_data_cube.shape=}\n" + \
      f"{radar_hdf5_controller.delta_time=}\n" + \
      f"{radar_hdf5_controller.shape=}\n" + \
      f"{radar_hdf5_controller.timestamp=}")

print(f"{frame_data_cube[:1,:,:,0]}")
